In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": { #local server is the name of the server, can be anything
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [11]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [19]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [20]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [21]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='7cef6bb2-f2b9-4bce-82ae-b625609651eb'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf040-0389-7373-bb63-dfc4aff9723c-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '28d10506-5282-4864-92a3-99b31b3c6cea', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 194, 'output_tokens': 22, 'total_tokens': 216, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "langchain-mcp-adapters library",\n  "follow_up_questions": null,\n  "answer": null,\n  "images":

## Online MCP

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [5]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [8]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
agent = create_agent(
    model=model,
    tools=tools
)

In [11]:
from langchain.messages import HumanMessage
from pprint import pprint
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='ec1bfbd4-e6cd-4503-8ba1-d2c09f6097c3'),
              AIMessage(content='I need to know which timezone you want me to get the time for. Can you please provide it?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf047-3bc7-7ee3-a931-5ff36eb3b214-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 250, 'output_tokens': 21, 'total_tokens': 271, 'input_token_details': {'cache_read': 0}})]}
